# 🚀 RAG 进阶：分块艺术与问题改写实战

在基础 RAG 链路中，我们通过“检索+生成”解决了模型知识过时的问题。但在实际工程中，面对高质量要求的业务场景，简单的拼接往往力有不逮。

本节实验将使用极端的 **“发疯文学”** 数据来测试 RAG 系统的极限，重点探讨：
1. **Chunking 颗粒度**：切分得太细或太粗会对检索结果产生什么影响？
2. **Query 改写 (Query Rewriting)**：用户问得模糊时，如何利用 LLM 自主优化检索词？
3. **进阶 Prompt 控制**：如何强制模型在面对“离奇事实”时保持高度忠实点。

---

## 🛠️ 1. 环境初始化：加载 RAG 引擎

我们将复用上一节的 CPU 优化模型组合，确保在本地环境流畅运行。

In [ ]:
import os
import torch
import torch.nn.functional as F
from pathlib import Path
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel, AutoModelForSequenceClassification
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# 1. 设置镜像站加速
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

def load_rag_models():
    """加载 RAG 全流程所需的本地模型"""
    models_root = Path("models")
    llm_id, emb_id, rerank_id = "Qwen/Qwen2-0.5B-Instruct", "BAAI/bge-small-zh-v1.5", "BAAI/bge-reranker-v2-m3"
    
    def quick_load(repo_id, model_type="llm"):
        model_name = repo_id.split("/")[-1]
        local_path = models_root / model_name
        snapshot_download(repo_id=repo_id, local_dir=str(local_path), local_dir_use_symlinks=False, ignore_patterns=["*.msgpack", "*.h5", "*.ot"])
        
        tokenizer = AutoTokenizer.from_pretrained(str(local_path))
        if model_type == "llm":
            model = AutoModelForCausalLM.from_pretrained(str(local_path), device_map="cpu")
        elif model_type == "emb":
            model = AutoModel.from_pretrained(str(local_path)).to("cpu")
        elif model_type == "rerank":
            model = AutoModelForSequenceClassification.from_pretrained(str(local_path)).to("cpu")
        return model, tokenizer

    (llm, llm_tok), (emb, emb_tok), (rerank, rerank_tok) = quick_load(llm_id, "llm"), quick_load(emb_id, "emb"), quick_load(rerank_id, "rerank")
    if llm_tok.pad_token is None: llm_tok.pad_token = llm_tok.eos_token
    return (llm, llm_tok), (emb, emb_tok), (rerank, rerank_tok)

(llm, llm_tok), (emb, emb_tok), (rerank, rerank_tok) = load_rag_models()
print("\n✅ RAG 进阶引擎模型加载完毕。")

## 🧪 2. 压力测试数据集：超越常识的“发疯文学”

我们选择了一段逻辑极其跳跃、完全不符合常规物理定律的文本。这将考验检索器是否能捕捉离奇的关联，以及生成器是否能放下“偏见”遵循资料。

In [ ]:
nonsense_doc = """意大利面就应该拌42号混凝土，因为这个螺丝钉的长度，它很容易会直接影响到挖掘机的扭矩你知道吧，你往里砸的时候，一瞬间它就会产生大量的高能蛋白，俗称UFO，会严重影响经济的发展，甚至对整个太平洋以及充电器都会造成一定的核污染，你知道吧？再者说，根据这个勾股定理，你可以很容易地推断出人工饲养的东条英机，它是可以捕获野生的三角函数的，所以说这个，你不管秦始皇的切面是否具有放射性啊，特朗普的N次方是否含有沉淀物啊，都不影响这个沃尔玛跟维尔康在南极会合。"""

print(f"🔥 压力测试文本长约 {len(nonsense_doc)} 字。")

## 🧱 2. 分块颗粒度的挑战 (The Granularity Challenge)

在 RAG 中，分块（Chunking）是对文本信息密度的重构。其核心在于寻找最适合检索器的语义窗口：
- **较细的分块 (Small Size)**：可能导致语境碎片化，虽然命中更精准，但可能丢失了逻辑链条。
- **较粗的分块 (Large Size)**：保留了完整的叙事背景，但由于包含过多非相关的“噪音”，可能会稀释检索得分。

In [ ]:
# 🛠️ 核心工具函数定义
def fixed_chunking(text, size=30, overlap=10):
    """基础固定长度分块函数"""
    return [text[i:i+size] for i in range(0, len(text), size-overlap)]

def get_embeddings(texts, model, tokenizer):
    """通用 Embedding 处理函数"""
    inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to("cpu")
    with torch.no_grad():
        outputs = model(**inputs)
        embeddings = outputs.last_hidden_state[:, 0, :]
        embeddings = F.normalize(embeddings, p=2, dim=1)
    return embeddings

# 为后续环节（如 Query 改写、Prompt 进阶）定义统一的基准分块
normal_chunks = fixed_chunking(nonsense_doc)
normal_embs = get_embeddings(normal_chunks, emb, emb_tok)

print(f"✅ 基础分块已准备（Size=30, overlap=10, 数量={len(normal_chunks)}），用于后续实验环节与 Rerank 展示。")

### 🔎 检索实验：观察不同 Size 对召回质量的影响
我们将通过梯度实验（Size = 4, 8, 12, 16, 20），对比在“搜索核心要素”时，不同分块粒度带出的“上下文全貌”有何差异。

In [ ]:
test_q = "什么会直接影响到挖掘机的扭矩？"
q_emb = get_embeddings([test_q], emb, emb_tok)

def demo_retrieval(q_e, c_list, c_embeddings, label=""):
    """执行检索并打印得分"""
    sims = F.cosine_similarity(q_e, c_embeddings, dim=1)
    score, idx = torch.topk(sims, k=1)
    if label: print(f"🔹 [{label}]")
    print(f"   🏆 [Top-1 检索结果] 命中内容: \"{c_list[idx.item()].strip()}\"")
    print(f"   📡 [相似度得分]: {score.item():.4f}")

sizes = [4, 8, 12, 16, 20]

print(f"🧪 实验进行中：Query = \"{test_q}\"\n")

for s in sizes:
    # 1. 物理切分
    current_chunks = fixed_chunking(nonsense_doc, size=s, overlap=0)
    
    # 2. 展示分块详情，用 | 隔开
    visualized = " | ".join([c.strip() for c in current_chunks])
    print(f"Size = {s} (生成 {len(current_chunks)} 个块)：")
    print(f"   | {visualized} |")
    
    # 3. 向量化并检索
    current_embs = get_embeddings(current_chunks, emb, emb_tok)
    demo_retrieval(q_emb, current_chunks, current_embs)
    print("-" * 80 + "\n")

### 🔄 颗粒度的补充：Overlap (重叠度) 实验

在文本信息划分时，最担心的就是核心逻辑恰好被切分在两个块的边缘，导致“头重脚轻”或语义丢失。利用 **Overlap** 可以让块与块之间保持一定的语义连续性。

我们将针对之前研究的每一个 Size，分别观察 **25%** 与 **50%** 重叠率下的分块密度变化及检索质量。

In [ ]:
test_q_overlap = "什么会直接影响到挖掘机的扭矩？"
q_emb_overlap = get_embeddings([test_q_overlap], emb, emb_tok)

overlap_ratios = [0.25, 0.5]
sizes = [4, 8, 12, 16, 20] # 沿用之前的梯度

print(f"🧪 Overlap 梯度实验持续进行中：对比 25% 与 50% 重叠对语义碎片的修复效果\n")

for s in sizes:
    for ratio in overlap_ratios:
        o_val = int(s * ratio)
        # 物理切分 (保证 overlap 不超过 size-1)
        current_chunks = fixed_chunking(nonsense_doc, size=s, overlap=min(o_val, s-1))
        
        # 展示分块详情，用 | 隔开
        visualized = " | ".join([c.strip() for c in current_chunks])
        print(f"📏 Size = {s}, Overlap = {int(ratio*100)}% ({o_val}字) -> 生成 {len(current_chunks)} 个块：")
        print(f"   | {visualized} |")
        
        # 向量化并检索
        current_embs = get_embeddings(current_chunks, emb, emb_tok)
        demo_retrieval(q_emb_overlap, current_chunks, current_embs)
        print("-" * 100)
    print("+" * 100 + "\n")

## 🤖 3. 问题改写：让检索更“聪明” (Query Rewriting)

有时候用户问得不清楚（如“那玩意儿拌啥？”），直接用向量检索会失效。此时需要利用 LLM 对问题进行重写。

### 🛠️ 核心策略：
1. **语义重写 (Semantic Rewriting)**：将口语化表述补全为专业表述。
2. **HyDE (假想文档)**：让 LLM 先写一个“假答案”，用假答案去搜真资料。

In [ ]:
def chat_rewrite_query(query, method="rewrite"):
    """根据选定方法重写查询词，引入‘背景参考’以提高匹配度"""
    # 在工业实践中，我们常会先通过关键词提取或摘要，给重写模型提供一个“知识图谱”或“主题参考”
    kb_keywords = ["42号混凝土", "挖掘机扭矩", "高能蛋白UFO", "勾股定理", "人工饲养的东条英机", "三角函数", "秦始皇的切面", "沃尔玛/维尔康"]
    
    if method == "rewrite":
        prompt = (f"你是一个检索词优化助手。\n"
                  f"我们的文档库主要涉及主题：{kb_keywords}\n"
                  f"请根据这些意象，将以下模糊问题改写为适合检索的完整描述性句子。即使问题很离奇，也请将其关联到上述主题。\n"
                  f"问题：{query}\n"
                  f"改写后的检索词：")
    elif method == "hyde":
        prompt = (f"请根据以下提示意象：{kb_keywords}，对问题写一个简短的‘假答案’用于检索。\n"
                  f"问题：{query}\n"
                  f"假答案：")
    
    messages = [{"role": "user", "content": prompt}]
    text = llm_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = llm_tok([text], return_tensors="pt").to("cpu")
    
    with torch.no_grad():
        generated_ids = llm.generate(
            **model_inputs, 
            max_new_tokens=60, 
            do_sample=False, 
            num_beams=1,
            pad_token_id=llm_tok.pad_token_id
        )
        response_ids = generated_ids[0][len(model_inputs.input_ids[0]):]
        result = llm_tok.decode(response_ids, skip_special_tokens=True).strip()
    
    return result.split("\n")[0]

# 使用您设计的强指向性模糊问题
fuzzy_q = "饿了，想吃"
rewritten_q = chat_rewrite_query(fuzzy_q, method="rewrite")
hyde_q = chat_rewrite_query(fuzzy_q, method="hyde")

print(f"🌀 原始模糊问题: {fuzzy_q}")
print(f"✨ 提示词重构后 (Rewrite): {rewritten_q}")
print(f"🧠 提示词重构后 (HyDE): {hyde_q}")

### ⚖️ 检索性能对比
对比模糊提问 vs 重写提问的相似度增益。

In [ ]:
print("📊 效果评测：")
demo_retrieval(get_embeddings([fuzzy_q], emb, emb_tok), normal_chunks, normal_embs, "原始模糊查询")
demo_retrieval(get_embeddings([rewritten_q], emb, emb_tok), normal_chunks, normal_embs, "LLM 重写查询")
demo_retrieval(get_embeddings([hyde_q], emb, emb_tok), normal_chunks, normal_embs, "HyDE 伪文档查询")

## ✍️ 4. 进阶提示词：约束与引导 (Advanced Prompting)

在面对不合常理的知识（如搅拌面条用混凝土）时，LLM 可能会由于预训练常识而拒绝认同资料。此时我们需要更强的 Prompt 约束。

In [ ]:
def advanced_rag_chat(query, contexts, style="strict"):
    context_text = "\n".join([f"[参考资料 {i+1}]: {c.strip()}" for i, c in enumerate(contexts)])
    
    if style == "strict":
        system_p = f""
        user_p = f"\n参考资料：\n{context_text}\n问题：{query}"
    elif style == "wild":
        system_p = f"你是一个发疯文学大师。请结合参考资料中的内容，用同样离奇的方式回答用户的问题。"
        user_p = f"\n参考资料：\n{context_text}\n问题：{query}"

    messages = [{"role": "system", "content": system_p}, {"role": "user", "content": user_p}]
    text = llm_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = llm_tok([text], return_tensors="pt").to("cpu")
    
    with torch.no_grad():
        generated_ids = llm.generate(**model_inputs, max_new_tokens=100, do_sample=True, temperature=0.7, pad_token_id=llm_tok.pad_token_id)
        response_ids = generated_ids[0][len(model_inputs.input_ids[0]):]
        result = llm_tok.decode(response_ids, skip_special_tokens=True).strip()
    
    return result

# 获取 Top-1 内容用于测试
q_target = "混凝土能吃吗？"
sims = F.cosine_similarity(get_embeddings([q_target], emb, emb_tok), normal_embs, dim=1)
best_idx = torch.argmax(sims).item()
best_context = [normal_chunks[best_idx]]

print(f"🔹 正常模式回答：\n{advanced_rag_chat(q_target, best_context, style='strict')}")
print(f"\n🔸 发疯文学模式回答：\n{advanced_rag_chat(q_target, best_context, style='wild')}")

### 🔎 深度解析：指令的“临门一脚”原则 (Recency Bias)

在 RAG 实践中，开发者常发现：**将约束条件放在 User 问题的前一刻，比放在 System 角色中效果更好。**

**主要原因：**
1. **近因效应 (Recency Bias)**：Decoder 架构模型对序列末尾的 Token 关注度最高。当参考资料 (Context) 非常长时，System 指令会被推到“记忆边缘”。
2. **SFT 训练分布**：主流模型在指令微调阶段，绝大多数关键指令都出现在 User 提问中。模型因此形成“重点看 User 指令”的直觉。
3. **RDQ 结构 (Rule-Data-Query)**：工业界推崇的 RAG 结构是：先把资料塞满，在资料之后、问句之前放下“最后通牒”。

In [ ]:
def advanced_rag_chat_v2(query, contexts, style="user"):
    context_text = "\n".join([f"[参考资料 {i+1}]: {c.strip()}" for i, c in enumerate(contexts)])
    
    if style == "system":
        system_p = f"是一个发疯文学大师。请结合参考资料：\n{context_text}，用同样离奇的方式回答用户的问题。"
        user_p = f"\n问题：{query}"
    elif style == "user":
        system_p = f"你是一个发疯文学大师。请结合参考资料中的内容，用同样离奇的方式回答用户的问题。"
        user_p = f"\n参考资料：\n{context_text}\n问题：{query}"

    messages = [{"role": "system", "content": system_p}, {"role": "user", "content": user_p}]
    text = llm_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = llm_tok([text], return_tensors="pt").to("cpu")
    
    with torch.no_grad():
        generated_ids = llm.generate(**model_inputs, max_new_tokens=100, do_sample=True, temperature=0.7, pad_token_id=llm_tok.pad_token_id)
        response_ids = generated_ids[0][len(model_inputs.input_ids[0]):]
        result = llm_tok.decode(response_ids, skip_special_tokens=True).strip()
    
    return result

# 获取 Top-1 内容用于测试
q_target = "混凝土能吃吗？"
sims = F.cosine_similarity(get_embeddings([q_target], emb, emb_tok), normal_embs, dim=1)
best_idx = torch.argmax(sims).item()
best_context = [normal_chunks[best_idx]]

print(f"🔹 System模式回答：\n{advanced_rag_chat_v2(q_target, best_context, style='system')}")
print(f"\n🔸 User模式回答：\n{advanced_rag_chat_v2(q_target, best_context, style='user')}")

## 📈 5. 本课知识复盘 (Summary)

1. **Chunking 颗粒度**：过细会导致语义由于断裂而失效（如无法联结“行为”与“后果”）。
2. **Query 改写**：解决用户偏离检索词、指代不明等问题的关键步骤。
3. **Prompt 权威性**：在垂直领域中，RAG 资料的权威性应通过 Prompt 强制高于 LLM 的预训练常识。